# Phase 6C - GraphSAGE Prototype

Notebook ini menjalankan GraphSAGE sebagai node classifier pada graph akun gabungan Batch 2 dan Batch 3. Node tidak berlabel tetap dipertahankan untuk message passing; loss hanya dihitung pada node berlabel di train split.

> Status hasil: `PROTOTYPE_ONLY`. Evaluasi transductive ini belum menggantikan cross-issue dan inductive holdout.

In [1]:
from pathlib import Path
import json
import sys

ROOT = next(path for path in [Path.cwd(), *Path.cwd().parents] if (path / 'PRD.md').exists())
SRC = ROOT / 'src'
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

print('Repository:', ROOT)
print('Prototype outputs:', ROOT / 'output' / 'phase6_prototypes')


Repository: /home/andry-alfisi/Project/Project/Buzzer-Detection-
Prototype outputs: /home/andry-alfisi/Project/Project/Buzzer-Detection-/output/phase6_prototypes


In [2]:
from models.graphsage_prototype import run_prototype

readiness = run_prototype(
    root=ROOT,
    hidden_size=32,
    epochs=120,
    learning_rate=0.01,
    device='cpu',
)
print(json.dumps(readiness, indent=2, ensure_ascii=False, default=str))


/home/andry-alfisi/anaconda3/lib/python3.13/site-packages/torch/jit/_script.py:1491: FutureWarning: `torch.jit.script` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(


=== GraphSAGE PROTOTYPE ===
n: 30
accuracy: 0.6666666666666666
precision_buzzer: 0.6666666666666666
recall_buzzer: 0.75
f1_buzzer: 0.7058823529411765
roc_auc: 0.7008928571428571
pr_auc: 0.766503847352338
confusion_matrix:
[[ 8  6]
 [ 4 12]]
{
  "status": "PROTOTYPE_ONLY",
  "model": "GraphSAGE account node classifier",
  "generated_at_utc": "2026-11-11T09:39:59.713851+00:00",
  "repository": "/home/andry-alfisi/Project/Project/Buzzer-Detection-",
  "batches": [
    "batch2",
    "batch3"
  ],
  "metrics": {
    "n": 30,
    "accuracy": 0.6666666666666666,
    "precision_buzzer": 0.6666666666666666,
    "recall_buzzer": 0.75,
    "f1_buzzer": 0.7058823529411765,
    "confusion_matrix": [
      [
        8,
        6
      ],
      [
        4,
        12
      ]
    ],
    "roc_auc": 0.7008928571428571,
    "pr_auc": 0.766503847352338
  },
  "artifacts": {
    "model": "/home/andry-alfisi/Project/Project/Buzzer-Detection-/output/phase6_prototypes/graphsage/model/graphsage_prototype.pt",

In [3]:
metrics = readiness['metrics']
assert readiness['status'] == 'PROTOTYPE_ONLY'
assert metrics['n'] > 0
assert readiness['data']['nodes'] > 0
assert readiness['data']['message_passing_edges'] > 0
assert Path(readiness['artifacts']['model']).exists()
assert Path(readiness['artifacts']['node_predictions']).exists()
assert Path(readiness['artifacts']['confusion_matrix']).exists()
print('GraphSAGE notebook validation: OK')
print('Graph stats:', {key: readiness['data'][key] for key in ('nodes', 'labeled_nodes', 'unlabeled_nodes', 'valid_edge_events', 'message_passing_edges')})


GraphSAGE notebook validation: OK
Graph stats: {'nodes': 2406, 'labeled_nodes': 100, 'unlabeled_nodes': 2306, 'valid_edge_events': 1365, 'message_passing_edges': 2730}


## Interpretasi

- Edge berasal dari reply/quote yang lolos validasi canonical.
- Reverse edge ditambahkan khusus untuk message passing agar informasi dapat mengalir dua arah; relasi asli tetap disimpan sebagai source-to-target pada data.
- Evaluasi berikutnya harus mencakup split berdasarkan isu dan edge holdout untuk menguji generalisasi serta mencegah leakage.